# Content-Yield Gate — Full Range (headless GPU)

Measures **accepted full-street flop decision records per root** at full ranges and
projects vs the 1,200-record / 30-session launch targets (PRD v1.3 §5.3/§9.4).

1. kaggle.com → **New Notebook** → **File → Import Notebook** (upload this).
2. **Settings → Accelerator → GPU T4**, Internet **On**.
3. **Save Version → Save & Run All (Commit)**, close the tab (~3 h headless).
4. Return → **last cell output** → copy `yield_report.json`.


In [ ]:
!nvidia-smi -L || echo 'NO GPU — Settings -> Accelerator -> GPU'

In [ ]:
import base64, zipfile, io, os
_ZIP_B64 = (
    "..."
)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_ZIP_B64))) as z: z.extractall('/kaggle/working/poker')
print('unpacked ->', sorted(os.listdir('/kaggle/working/poker')))

In [ ]:
try:
    import cupy as cp
except Exception:
    import subprocess,sys; subprocess.run([sys.executable,'-m','pip','install','-q','cupy-cuda12x']); import cupy as cp
nm=cp.cuda.runtime.getDeviceProperties(0)['name']; print('cupy',cp.__version__,'|',nm.decode() if isinstance(nm,bytes) else nm)

In [ ]:
# Smoke: 1 root at full range (~15 min) — confirms pipeline
!cd /kaggle/working/poker && PYTHONPATH=src python -m pokertrainer.content_yield \
    --solver gpu --dtype float64 --n 400 --iters 400 --roots 0 --out /kaggle/working/cy_smoke

## Full run — 12 roots at full ranges (~3 h headless)


In [ ]:
!cd /kaggle/working/poker && PYTHONPATH=src python -m pokertrainer.content_yield \
    --solver gpu --dtype float64 --n 400 --iters 400 --out /kaggle/working/cyout

In [ ]:
import json; print(json.dumps(json.load(open('/kaggle/working/cyout/yield_report.json')), indent=1))

_Full record set: `/kaggle/working/cyout/records.json` (Output tab). Paste the `yield_report.json` back._